In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader

# Verify hardware acceleration and library versions
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Define execution device (GPU preferred for Bi-GRU operations)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
from google.colab import drive
# Mount Google Drive to access pre-extracted features (.npz files)
drive.mount('/content/drive')

In [ ]:
class AnomalyDetector(nn.Module):
    def __init__(self, input_size=2134, hidden_size=256, num_classes=14):
        super(AnomalyDetector, self).__init__()

        # 1. Temporal Encoder: Processes fused I3D (2048) + YOLO (86) features
        self.bigru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Bi-directional output is concatenated (hidden_size * 2)
        combined_dim = hidden_size * 2

        # 2. MIL Ranking Head: Produces a [0, 1] anomaly score per segment
        self.anomaly_head = nn.Sequential(
            nn.Linear(combined_dim, 1),
            nn.Sigmoid()
        )

        # 3. Multi-Class Classification Head: Identifies specific crime category
        self.class_head = nn.Sequential(
            nn.Linear(combined_dim, num_classes),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        # x shape: [Batch, Segments, 2134]
        gru_out, _ = self.bigru(x)

        # Branch A: Regression-like score for temporal localization
        anomaly_scores = self.anomaly_head(gru_out)

        # Branch B: Fine-grained classification probabilities
        class_probs = self.class_head(gru_out)

        return anomaly_scores, class_probs

In [ ]:
feature_dir = '/content/drive/MyDrive/UCF_Crime/features'
feature_files = [f for f in os.listdir(feature_dir) if f.endswith('.npz')]

all_features = []
all_labels = []

print(f"Loading features from: {feature_dir}...")

for filename in feature_files:
    file_path = os.path.join(feature_dir, filename)
    data = np.load(file_path, allow_pickle=True)

    # Load features: Shape [Segments, 2134]
    features = data['features']

    # Extract metadata dictionary to get integer label
    metadata = data['metadata'].item()
    label = int(metadata['label'])

    all_features.append(torch.FloatTensor(features))
    all_labels.append(label)

print(f"Successfully loaded {len(all_features)} video feature sets.")

In [ ]:
class VideoDataset(Dataset):
    """
    Custom Dataset for handling pre-extracted video features.
    Note: Videos may have different numbers of segments.
    """
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        # Returns a tuple: (Tensor[Segments, 2134], Label)
        return self.features[idx], self.labels[idx]

# Helper to handle variable sequence lengths in a single batch
def collate_fn(batch):
    features, labels = zip(*batch)
    # Pads sequences to the length of the longest video in the batch
    features_padded = nn.utils.rnn.pad_sequence(features, batch_first=True)
    labels = torch.LongTensor(labels)
    return features_padded, labels

In [ ]:
class MILRankingLoss(nn.Module):
    def __init__(self, lambda1=8e-5, lambda2=8e-5):
        super(MILRankingLoss, self).__init__()
        self.lambda1 = lambda1  # Temporal Smoothness weight
        self.lambda2 = lambda2  # Sparsity weight

    def forward(self, anomaly_scores, labels):
        anomaly_scores = anomaly_scores.squeeze(-1)

        # Identify Anomalous vs Normal videos in the current batch
        pos_mask = (labels > 0)
        neg_mask = (labels == 0)

        # Safety check for MIL ranking requirement
        if not pos_mask.any() or not neg_mask.any():
            return torch.tensor(0.0, device=anomaly_scores.device, requires_grad=True)

        # MIL Ranking: Compare max score of Anomaly video vs max score of Normal video
        max_scores, _ = torch.max(anomaly_scores, dim=1)
        max_anomaly = max_scores[pos_mask].mean()
        max_normal = max_scores[neg_mask].mean()

        # Hinge loss to enforce a margin of 1.0
        ranking_loss = torch.relu(1.0 - max_anomaly + max_normal)

        # Smoothness: Penalize large score jumps between consecutive segments
        diff = anomaly_scores[:, 1:] - anomaly_scores[:, :-1]
        smoothness = torch.sum(diff**2)

        # Sparsity: Enforce that anomaly segments are rare (short-duration)
        sparsity = torch.sum(anomaly_scores[pos_mask])

        return ranking_loss + (self.lambda1 * smoothness) + (self.lambda2 * sparsity)

In [ ]:
# 1. Initialization
batch_size = 32
num_epochs = 100

dataset = VideoDataset(all_features, all_labels)
# Added collate_fn to handle varying segment lengths
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

model = AnomalyDetector().to(device)

# Adadelta is robust for MIL tasks involving sparse gradients
optimizer = optim.Adadelta(model.parameters(), lr=1.0, rho=0.9, eps=1e-6)

# Dual-head loss functions
criterion_mil = MILRankingLoss()
criterion_class = nn.CrossEntropyLoss()

# 2. Training Loop
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)

        # Forward pass: Generate detection scores and classification logits
        anomaly_scores, class_logits = model(features)

        # Multi-task loss calculation
        loss_ranking = criterion_mil(anomaly_scores, labels)

        # Flatten time dimension for cross-entropy: [Batch*Segments, Classes]
        loss_classification = criterion_class(class_logits.view(-1, 14),
                                              labels.repeat_interleave(features.size(1)))

        total_loss = loss_ranking + loss_classification

        # Optimization step
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        epoch_loss += total_loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_loader):.4f}")

In [ ]:
import matplotlib.pyplot as plt

def visualize_anomaly(model, video_features, video_name="Test Video"):
    """
    Predicts and plots anomaly scores across all segments of a video.
    """
    model.eval()
    with torch.no_grad():
        # video_features shape: [Segments, 2134] -> add batch dim: [1, Segments, 2134]
        input_tensor = video_features.unsqueeze(0).to(device)

        # Get predictions
        anomaly_scores, class_probs = model(input_tensor)

        # Convert to numpy for plotting
        scores = anomaly_scores.squeeze().cpu().numpy()

        # Get the predicted class (max probability from the class head)
        # We take the mean across all segments to get a video-level classification
        mean_class_probs = class_probs.squeeze().mean(dim=0)
        pred_class_idx = torch.argmax(mean_class_probs).item()

        # UCF-Crime Class Mapping (Update this list based on your specific labels)
        classes = ['Normal', 'Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary',
                   'Explosion', 'Fighting', 'Robbery', 'Shooting', 'Shoplifting',
                   'Stealing', 'Vandalism', 'RoadAccidents']

        # Plotting
        plt.figure(figsize=(12, 5))
        plt.plot(scores, label='Anomaly Score', color='red', linewidth=2)
        plt.fill_between(range(len(scores)), scores, color='red', alpha=0.2)

        plt.title(f"Anomaly Detection: {video_name}\nPredicted Class: {classes[pred_class_idx]}")
        plt.xlabel("Video Segments (Time)")
        plt.ylabel("Anomaly Probability (0-1)")
        plt.ylim(0, 1.1)
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        plt.show()

# --- Example Usage ---
# Pick a random video from your loaded features
idx = 0
test_feat = all_features[idx]
visualize_anomaly(model, test_feat, video_name=feature_files[idx])